# Part 1 - Baseline Image Captioning Model (CNN Encoder + LSTM Decoder)
### Neural Networks & Deep Learning - Final Project

This notebook implements **Part 1** of the project: a baseline image captioning model consisting of a
pretrained CNN image encoder and an LSTM caption decoder, trained end-to-end (decoder only) and evaluated
with BLEU-1 / BLEU-4.

## Inputs required (add both as Kaggle Datasets via *Add Input*)

1. The **preprocessed artifacts** produced by the previous preprocessing notebook:
   `manifest.json`, `vocab.json`, `captions.npz`, `splits.json`, `features.h5`, `feature_index.json`,
   `transform_config.json`.
2. The **original raw image dataset** (Flickr8k / Flickr30k) - used *only* for qualitative visualization
   (showing the input image next to its generated caption), never for training.

## Design choices, grounded in the reviewed similar projects

- **Encoder:** a frozen pretrained CNN (ResNet50) whose spatial feature maps were already extracted and
  cached to `features.h5` in the preprocessing notebook. This notebook never runs the CNN forward pass -
  it only reads small cached feature vectors. This mirrors the standard approach used in essentially every
  reference implementation reviewed (VGG16 / InceptionV3 / ResNet50 feature caching before LSTM training),
  and is the single biggest lever against Kaggle OOM / time-limit issues.
- **Decoder:** a single-layer LSTM using a *merge* architecture - the (mean-pooled, projected) image
  feature vector is concatenated with the word embedding at every decoding step, matching the pattern used
  in several of the reviewed Kaggle/GitHub baselines (image embedding concatenated with word embedding,
  fed to the LSTM to predict the next word).
- **Loss:** token-level cross-entropy with `ignore_index` on the `<pad>` token.

## Kaggle-limit safety measures implemented in this notebook

| Risk | Mitigation |
|---|---|
| GPU OOM from running the CNN every batch | CNN is never invoked here - only cached feature vectors are read |
| GPU OOM from large batches / fp32 activations | Automatic Mixed Precision (`torch.cuda.amp`) + moderate batch size |
| Exploding gradients destabilizing LSTM training | Gradient clipping (`clip_grad_norm_`) |
| Losing all progress if the 9-12h Kaggle session ends | Checkpoint saved every epoch + automatic resume from checkpoint |
| Wasting the whole time budget on a training run that stops mid-epoch | Wall-clock time budget tracked every epoch, training stops gracefully with a safety margin |
| Wasting GPU time after the model has converged | Early stopping on validation loss / BLEU with a patience counter |
| Expensive BLEU decoding every epoch | BLEU computed only every few epochs on a small validation subset |
| Kernel timeout on long silent cells | Periodic heartbeat logging inside every long-running loop |
| Blowing the `/kaggle/working` file/size quota | All artifacts written to a handful of consolidated files, not thousands of small files |


In [ ]:
# Install / verify required packages (most are preinstalled on Kaggle)
import importlib, subprocess, sys

REQUIRED = {
    "h5py": "h5py",
    "nltk": "nltk",
    "tqdm": "tqdm",
    "torch": "torch",
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "Pillow",
    "matplotlib": "matplotlib",
    "psutil": "psutil",
}
missing = []
for import_name, pip_name in REQUIRED.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
else:
    print("All required packages are already available.")


In [ ]:
import os, json, gc, time, random, math, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import h5py
import psutil
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def report_resources():
    vm = psutil.virtual_memory()
    target = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
    disk_free = shutil.disk_usage(target).free
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"RAM total: {vm.total / 1e9:.1f} GB | available: {vm.available / 1e9:.1f} GB")
    print(f"Disk free (working dir): {disk_free / 1e9:.1f} GB")


report_resources()


## 1. Configuration

Every hyperparameter required by the project report (learning rate, batch size, optimizer, encoder/decoder
type, embedding/hidden dimensions, vocabulary size) is defined here in one place and is later exported as
a hyperparameter table.


In [ ]:
class CFG:
    SEED = 42

    # ---- model ----
    ENCODER_NAME = "resnet50 (frozen, features precomputed)"
    DECODER_NAME = "single-layer LSTM (merge architecture)"
    EMBED_DIM = 256
    HIDDEN_DIM = 512
    NUM_LAYERS = 1
    DROPOUT = 0.5

    # ---- optimization ----
    BATCH_SIZE = 128
    LR = 3e-4
    WEIGHT_DECAY = 1e-5
    OPTIMIZER_NAME = "Adam"
    GRAD_CLIP = 5.0
    MAX_EPOCHS = 30
    PATIENCE = 5                 # early-stopping patience, in BLEU evaluation rounds
    NUM_WORKERS = 2

    # ---- Kaggle safety ----
    TIME_BUDGET_HOURS = 8.0      # keep comfortably under the 9-12h Kaggle session cap
    TIME_SAFETY_MARGIN_MIN = 15  # stop early if less than this many minutes remain in the budget

    # ---- evaluation ----
    BLEU_EVAL_EVERY = 2          # compute validation BLEU every N epochs (expensive: needs decoding)
    BLEU_EVAL_SUBSET = 200       # number of validation images used for the periodic BLEU check
    GEN_MAX_LEN = None           # filled in later from MAX_LEN in the manifest

    # ---- output ----
    OUTPUT_DIR = "/kaggle/working/part1_baseline"
    LAST_CKPT = "last_model.pt"
    BEST_CKPT = "best_model.pt"


set_seed(CFG.SEED)
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print({k: v for k, v in vars(CFG).items() if not k.startswith("_")})


## 2. Locate input artifacts

`/kaggle/input` is scanned automatically for the preprocessing outputs and for the raw image folder, so no
hard-coded dataset slug is required.


In [ ]:
def find_preprocessed_dir(root="/kaggle/input"):
    for dirpath, _, filenames in os.walk(root):
        if "manifest.json" in filenames and "features.h5" in filenames:
            return dirpath
    return None


def find_images_dir(root="/kaggle/input"):
    best_dir, best_count = None, 0
    for dirpath, _, filenames in os.walk(root):
        n = sum(1 for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png")))
        if n > best_count:
            best_count, best_dir = n, dirpath
    return best_dir, best_count


PREP_DIR = find_preprocessed_dir()
RAW_IMAGES_DIR, n_raw_images = find_images_dir()

assert PREP_DIR is not None, "Preprocessed artifacts not found. Add the preprocessing notebook output as an input dataset."
assert RAW_IMAGES_DIR is not None, "Raw image folder not found. Add the original Flickr8k/Flickr30k dataset as an input."

print("Preprocessed artifacts directory:", PREP_DIR)
print("Raw images directory:", RAW_IMAGES_DIR, "| images found:", n_raw_images)


In [ ]:
with open(os.path.join(PREP_DIR, "manifest.json")) as f:
    manifest = json.load(f)
with open(os.path.join(PREP_DIR, "vocab.json"), encoding="utf-8") as f:
    vocab = json.load(f)
with open(os.path.join(PREP_DIR, "feature_index.json")) as f:
    feature_index = json.load(f)

word2idx = vocab["word2idx"]
idx2word = vocab["idx2word"]
PAD_ID = vocab["special_tokens"]["pad"]
START_ID = vocab["special_tokens"]["start"]
END_ID = vocab["special_tokens"]["end"]
UNK_ID = vocab["special_tokens"]["unk"]

VOCAB_SIZE = len(idx2word)
MAX_LEN = manifest["max_caption_len"]
FEAT_DIM = feature_index["feat_dim"]
NUM_TOKENS = feature_index["num_tokens"]
id2idx = feature_index["id2idx"]

CFG.GEN_MAX_LEN = MAX_LEN

FEATURES_H5_PATH = os.path.join(PREP_DIR, "features.h5")
CAPTIONS_NPZ_PATH = os.path.join(PREP_DIR, "captions.npz")

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Max caption length: {MAX_LEN}")
print(f"Feature dim: {FEAT_DIM} | tokens per image: {NUM_TOKENS}")
print(f"Split sizes (images): {manifest['split_sizes_images']}")


## 3. Dataset and DataLoader

Reads the cached image features and the pre-tokenized captions directly - no image decoding, no CNN
forward pass. The HDF5 file handle is opened lazily *inside each worker process* (after the DataLoader
fork), which is the standard safe pattern for combining `h5py` with `num_workers > 0`.


In [ ]:
class CaptionFeatureDataset(Dataset):
    def __init__(self, npz_path, h5_path, id2idx, split):
        data = np.load(npz_path, allow_pickle=True)
        mask = data["split"] == split
        self.images = data["image"][mask]
        self.token_ids = data["token_ids"][mask].astype(np.int64)
        self.raw_captions = data["raw_caption"][mask]
        self.clean_captions = data["clean_caption"][mask]
        self.h5_path = h5_path
        self.id2idx = id2idx
        self._h5 = None

    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        h5 = self._get_h5()
        img_name = str(self.images[idx])
        row = self.id2idx[img_name]
        feat = h5["features"][row]                       # (num_tokens, feat_dim), float16
        feat = torch.from_numpy(feat.astype(np.float32)).mean(dim=0)  # pooled -> (feat_dim,)
        tokens = torch.from_numpy(self.token_ids[idx])
        return feat, tokens


def build_dataloader(split, batch_size, shuffle, num_workers):
    ds = CaptionFeatureDataset(CAPTIONS_NPZ_PATH, FEATURES_H5_PATH, id2idx, split)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"), drop_last=False, persistent_workers=(num_workers > 0),
    )
    return ds, loader


train_ds, train_loader = build_dataloader("train", CFG.BATCH_SIZE, True, CFG.NUM_WORKERS)
val_ds, val_loader = build_dataloader("val", CFG.BATCH_SIZE, False, CFG.NUM_WORKERS)
test_ds, test_loader = build_dataloader("test", CFG.BATCH_SIZE, False, CFG.NUM_WORKERS)

print(f"train captions: {len(train_ds)} | val captions: {len(val_ds)} | test captions: {len(test_ds)}")

# quick sanity check on shapes before committing to a full training run
sample_feat, sample_tokens = train_ds[0]
print("Sample feature shape:", sample_feat.shape, "| sample token ids shape:", sample_tokens.shape)


## 4. Model - CNN feature (frozen, precomputed) + LSTM decoder

The image feature is projected to `EMBED_DIM` and concatenated with the word embedding at every decoding
step (merge architecture). Training uses teacher forcing on the full fixed-length padded sequence; the loss
ignores `<pad>` positions.


In [ ]:
class CaptionLSTM(nn.Module):
    def __init__(self, vocab_size, feat_dim, embed_dim, hidden_dim, num_layers, dropout, pad_id):
        super().__init__()
        self.image_proj = nn.Linear(feat_dim, embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=embed_dim * 2, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, img_feat, input_tokens):
        # img_feat: (B, feat_dim) ; input_tokens: (B, T)
        img_embed = torch.tanh(self.image_proj(img_feat))               # (B, E)
        word_embed = self.embedding(input_tokens)                       # (B, T, E)
        img_embed_rep = img_embed.unsqueeze(1).expand(-1, word_embed.size(1), -1)
        lstm_in = torch.cat([word_embed, img_embed_rep], dim=-1)        # (B, T, 2E)
        lstm_out, _ = self.lstm(lstm_in)
        lstm_out = self.dropout(lstm_out)
        logits = self.fc(lstm_out)                                      # (B, T, V)
        return logits

    @torch.no_grad()
    def generate(self, img_feat, start_id, end_id, max_len, device):
        self.eval()
        B = img_feat.size(0)
        img_embed = torch.tanh(self.image_proj(img_feat)).unsqueeze(1)  # (B, 1, E)
        input_tok = torch.full((B, 1), start_id, dtype=torch.long, device=device)
        hidden = None
        generated = torch.zeros(B, max_len, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for t in range(max_len):
            word_embed = self.embedding(input_tok)                      # (B, 1, E)
            lstm_in = torch.cat([word_embed, img_embed], dim=-1)
            out, hidden = self.lstm(lstm_in, hidden)
            logits = self.fc(out.squeeze(1))                            # (B, V)
            next_tok = logits.argmax(dim=-1)                            # (B,)
            next_tok = torch.where(finished, torch.full_like(next_tok, end_id), next_tok)
            generated[:, t] = next_tok
            finished = finished | (next_tok == end_id)
            input_tok = next_tok.unsqueeze(1)
            if finished.all():
                break
        return generated


model = CaptionLSTM(
    vocab_size=VOCAB_SIZE, feat_dim=FEAT_DIM, embed_dim=CFG.EMBED_DIM,
    hidden_dim=CFG.HIDDEN_DIM, num_layers=CFG.NUM_LAYERS, dropout=CFG.DROPOUT, pad_id=PAD_ID,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
approx_mb = n_params * 4 / 1e6
print(f"Total parameters: {n_params:,} | trainable: {n_trainable:,} | approx size (fp32): {approx_mb:.1f} MB")
print(model)


## 5. Training utilities

Includes: cross-entropy loss with padding ignored, Adam optimizer, automatic mixed precision, gradient
clipping, checkpoint save/resume, and a small helper to build the multi-reference caption dictionary
needed for BLEU.


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


def save_checkpoint(path, epoch, best_val_bleu, history):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_val_bleu": best_val_bleu,
        "history": history,
        "cfg": {k: v for k, v in vars(CFG).items() if not k.startswith("_")},
    }, path)


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    return ckpt["epoch"], ckpt["best_val_bleu"], ckpt["history"]


def ids_to_words(ids):
    words = []
    for i in ids:
        i = int(i)
        if i == END_ID:
            break
        if i in (PAD_ID, START_ID):
            continue
        words.append(idx2word[i])
    return words


def build_reference_dict(dataset):
    refs = defaultdict(list)
    for img_name, clean_cap in zip(dataset.images, dataset.clean_captions):
        refs[str(img_name)].append(str(clean_cap).split())
    return refs


In [ ]:
def train_one_epoch(loader, log_every=50):
    model.train()
    total_loss, total_tokens = 0.0, 0
    t0 = time.time()
    for i, (feat, tokens) in enumerate(loader):
        feat = feat.to(DEVICE, non_blocking=True)
        tokens = tokens.to(DEVICE, non_blocking=True)
        inp = tokens[:, :-1]
        target = tokens[:, 1:]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(feat, inp)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        n_tok = (target != PAD_ID).sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok

        if (i + 1) % log_every == 0:
            print(f"  [train batch {i + 1}/{len(loader)}] running loss: {total_loss / max(total_tokens, 1):.4f} "
                  f"| elapsed: {(time.time() - t0) / 60:.1f} min", flush=True)

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_loss(loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for feat, tokens in loader:
        feat = feat.to(DEVICE, non_blocking=True)
        tokens = tokens.to(DEVICE, non_blocking=True)
        inp = tokens[:, :-1]
        target = tokens[:, 1:]
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(feat, inp)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), target.reshape(-1))
        n_tok = (target != PAD_ID).sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok
    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_bleu(dataset, references, max_images=None, batch_size=128):
    model.eval()
    unique_images = list(references.keys())
    if max_images is not None and len(unique_images) > max_images:
        rng = random.Random(CFG.SEED)
        unique_images = rng.sample(unique_images, max_images)

    hyps, refs_list = [], []
    h5 = h5py.File(FEATURES_H5_PATH, "r")
    for start in range(0, len(unique_images), batch_size):
        batch_names = unique_images[start:start + batch_size]
        feats = [h5["features"][id2idx[name]].astype(np.float32).mean(axis=0) for name in batch_names]
        feats = torch.from_numpy(np.stack(feats)).to(DEVICE)

        gen_ids = model.generate(feats, START_ID, END_ID, CFG.GEN_MAX_LEN, DEVICE)
        for row_idx, name in enumerate(batch_names):
            hyp_words = ids_to_words(gen_ids[row_idx].tolist())
            hyps.append(hyp_words)
            refs_list.append(references[name])
    h5.close()

    smoothing = SmoothingFunction().method1
    bleu1 = corpus_bleu(refs_list, hyps, weights=(1.0, 0, 0, 0), smoothing_function=smoothing)
    bleu4 = corpus_bleu(refs_list, hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
    return bleu1, bleu4, hyps, refs_list, unique_images


## 6. Resume-safe, time-budget-aware training loop

Before starting, the notebook checks whether a checkpoint from a previous (possibly interrupted) session
already exists and resumes from it. Training stops when any of the following happens: `MAX_EPOCHS` is
reached, validation BLEU-1 stops improving for `PATIENCE` evaluation rounds, or the wall-clock time budget
is about to run out.


In [ ]:
val_refs = build_reference_dict(val_ds)

last_ckpt_path = os.path.join(CFG.OUTPUT_DIR, CFG.LAST_CKPT)
best_ckpt_path = os.path.join(CFG.OUTPUT_DIR, CFG.BEST_CKPT)

history = {"train_loss": [], "val_loss": [], "val_bleu1": {}}
start_epoch = 0
best_val_bleu = -1.0
epochs_without_improvement = 0
epoch = start_epoch - 1

if os.path.exists(last_ckpt_path):
    start_epoch, best_val_bleu, history = load_checkpoint(last_ckpt_path)
    start_epoch += 1
    print(f"Resumed from checkpoint at epoch {start_epoch} (best val BLEU-1 so far: {best_val_bleu:.4f})")
else:
    print("No existing checkpoint found. Starting training from scratch.")

training_start_time = time.time()
budget_seconds = CFG.TIME_BUDGET_HOURS * 3600
margin_seconds = CFG.TIME_SAFETY_MARGIN_MIN * 60
epoch_durations = []

for epoch in range(start_epoch, CFG.MAX_EPOCHS):
    epoch_t0 = time.time()
    train_loss = train_one_epoch(train_loader)
    val_loss = evaluate_loss(val_loader)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    msg = f"Epoch {epoch + 1}/{CFG.MAX_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}"

    run_bleu = ((epoch + 1) % CFG.BLEU_EVAL_EVERY == 0) or (epoch == CFG.MAX_EPOCHS - 1)
    if run_bleu:
        bleu1, bleu4, *_ = evaluate_bleu(val_ds, val_refs, max_images=CFG.BLEU_EVAL_SUBSET)
        history["val_bleu1"][str(epoch)] = bleu1
        msg += f" | val_BLEU1={bleu1:.4f} | val_BLEU4={bleu4:.4f}"
        if bleu1 > best_val_bleu:
            best_val_bleu = bleu1
            epochs_without_improvement = 0
            save_checkpoint(best_ckpt_path, epoch, best_val_bleu, history)
            msg += " | new best model saved"
        else:
            epochs_without_improvement += 1

    epoch_dt = time.time() - epoch_t0
    epoch_durations.append(epoch_dt)
    avg_epoch_dt = sum(epoch_durations) / len(epoch_durations)
    elapsed = time.time() - training_start_time
    remaining_budget = budget_seconds - elapsed

    print(msg + f" | epoch time: {epoch_dt / 60:.1f} min | elapsed total: {elapsed / 60:.1f} min", flush=True)

    save_checkpoint(last_ckpt_path, epoch, best_val_bleu, history)

    if run_bleu and epochs_without_improvement >= CFG.PATIENCE:
        print(f"Early stopping: no BLEU-1 improvement for {CFG.PATIENCE} evaluation rounds.")
        break

    if remaining_budget < (avg_epoch_dt + margin_seconds):
        print("Stopping: not enough time budget left for another safe epoch. "
              "Re-run this notebook to resume training from the saved checkpoint.")
        break

print("Training loop finished.")
if not os.path.exists(best_ckpt_path):
    # ensure a best checkpoint always exists, even if BLEU was never evaluated (e.g. a very short run)
    save_checkpoint(best_ckpt_path, epoch, best_val_bleu, history)


## 7. Loss curve


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss (per token)")
plt.title("Part 1 Baseline - Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
loss_curve_path = os.path.join(CFG.OUTPUT_DIR, "loss_curve.png")
plt.savefig(loss_curve_path, dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", loss_curve_path)


## 8. Final evaluation on the test set

The best checkpoint (selected by validation BLEU-1) is reloaded and evaluated once on the full test set.


In [ ]:
_ = load_checkpoint(best_ckpt_path)
model.eval()

test_refs = build_reference_dict(test_ds)
test_bleu1, test_bleu4, test_hyps, test_refs_list, test_image_order = evaluate_bleu(
    test_ds, test_refs, max_images=None, batch_size=CFG.BATCH_SIZE,
)

print(f"Test BLEU-1: {test_bleu1:.4f}")
print(f"Test BLEU-4: {test_bleu4:.4f}")

metrics = {"test_bleu1": test_bleu1, "test_bleu4": test_bleu4, "num_test_images": len(test_image_order)}
with open(os.path.join(CFG.OUTPUT_DIR, "test_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=1)


## 9. Qualitative sample generations

A grid of test images with their ground-truth caption and the model's generated caption, as required by
the project deliverables.


In [ ]:
import textwrap

rng = random.Random(CFG.SEED)
sample_idx = rng.sample(range(len(test_image_order)), min(9, len(test_image_order)))

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for ax, idx in zip(axes.flat, sample_idx):
    img_name = test_image_order[idx]
    gt = " ".join(test_refs_list[idx][0])
    hyp = " ".join(test_hyps[idx]) if test_hyps[idx] else "(empty)"

    img_path = os.path.join(RAW_IMAGES_DIR, img_name)
    if not os.path.exists(img_path):
        # fall back to a recursive search if the raw dataset uses nested subfolders
        matches = list(Path(RAW_IMAGES_DIR).rglob(img_name))
        img_path = str(matches[0]) if matches else None

    if img_path is not None:
        with Image.open(img_path) as im:
            ax.imshow(im.convert("RGB"))
    ax.axis("off")
    title = "GT: " + textwrap.fill(gt, 30) + "\nGen: " + textwrap.fill(hyp, 30)
    ax.set_title(title, fontsize=8)

plt.tight_layout()
samples_path = os.path.join(CFG.OUTPUT_DIR, "test_sample_generations.png")
plt.savefig(samples_path, dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", samples_path)


## 10. Error analysis

Automated diagnostics to support the required discussion: word repetition, early/late truncation, and a
comparison of BLEU-1 on "simple" vs "complex" images (proxied by ground-truth caption length).


In [ ]:
def repetition_ratio(words):
    if len(words) == 0:
        return 0.0
    unique = len(set(words))
    return 1.0 - unique / len(words)


def per_sample_bleu1(hyp, refs):
    smoothing = SmoothingFunction().method1
    return corpus_bleu([refs], [hyp], weights=(1.0, 0, 0, 0), smoothing_function=smoothing)


rows = []
for name, hyp, refs in zip(test_image_order, test_hyps, test_refs_list):
    rows.append({
        "image": name,
        "generated": " ".join(hyp),
        "ground_truth": " ".join(refs[0]),
        "gen_length": len(hyp),
        "gt_length": len(refs[0]),
        "repetition_ratio": repetition_ratio(hyp),
        "bleu1": per_sample_bleu1(hyp, refs),
    })
analysis_df = pd.DataFrame(rows)

median_gt_len = analysis_df["gt_length"].median()
analysis_df["complexity"] = np.where(analysis_df["gt_length"] <= median_gt_len, "simple", "complex")

print("Average BLEU-1 by proxy complexity (based on ground-truth caption length):")
print(analysis_df.groupby("complexity")["bleu1"].mean())

print("\nShare of generated captions with noticeable word repetition (ratio > 0.3):",
      f"{(analysis_df['repetition_ratio'] > 0.3).mean() * 100:.1f}%")
print("Share of generated captions that hit the max length without producing <end>:",
      f"{(analysis_df['gen_length'] >= CFG.GEN_MAX_LEN - 1).mean() * 100:.1f}%")

worst = analysis_df.sort_values("bleu1").head(5)
best = analysis_df.sort_values("bleu1", ascending=False).head(5)
print("\nWorst 5 examples by BLEU-1:")
print(worst[["image", "ground_truth", "generated", "bleu1"]].to_string(index=False))
print("\nBest 5 examples by BLEU-1:")
print(best[["image", "ground_truth", "generated", "bleu1"]].to_string(index=False))

analysis_csv_path = os.path.join(CFG.OUTPUT_DIR, "error_analysis.csv")
analysis_df.to_csv(analysis_csv_path, index=False)
print("\nSaved:", analysis_csv_path)


## 11. Hyperparameter table and final manifest

A single table with exactly the fields requested in the project deliverables (learning rate, batch size,
epochs, optimizer, encoder/decoder type, embedding/hidden dimensions, vocabulary size), plus a manifest of
every output file with a check against Kaggle's `/kaggle/working` size and file-count limits.


In [ ]:
hparams = {
    "Learning Rate": CFG.LR,
    "Batch Size": CFG.BATCH_SIZE,
    "Epochs Trained": len(history["train_loss"]),
    "Max Epochs Configured": CFG.MAX_EPOCHS,
    "Optimizer": CFG.OPTIMIZER_NAME,
    "Encoder": CFG.ENCODER_NAME,
    "Decoder": CFG.DECODER_NAME,
    "Embedding Dimension": CFG.EMBED_DIM,
    "Hidden Dimension": CFG.HIDDEN_DIM,
    "Vocabulary Size": VOCAB_SIZE,
    "Dropout": CFG.DROPOUT,
    "Gradient Clip Norm": CFG.GRAD_CLIP,
    "Test BLEU-1": round(test_bleu1, 4),
    "Test BLEU-4": round(test_bleu4, 4),
}
hparams_df = pd.DataFrame(list(hparams.items()), columns=["Hyperparameter", "Value"])
print(hparams_df.to_string(index=False))

hparams_csv_path = os.path.join(CFG.OUTPUT_DIR, "hyperparameters_table.csv")
hparams_df.to_csv(hparams_csv_path, index=False)

out_files = [p for p in Path(CFG.OUTPUT_DIR).rglob("*") if p.is_file()]
total_size_gb = sum(p.stat().st_size for p in out_files) / 1e9
print(f"\nOutput files: {len(out_files)} (Kaggle soft limit is about 500)")
print(f"Total output size: {total_size_gb:.3f} GB (Kaggle soft limit is about 20 GB)")
for p in sorted(out_files):
    print(f"  - {p.relative_to(CFG.OUTPUT_DIR)}: {p.stat().st_size / 1e6:.2f} MB")

final_manifest = {
    "hyperparameters": hparams,
    "test_metrics": metrics,
    "files": {p.name: str(p.relative_to(CFG.OUTPUT_DIR)) for p in out_files},
}
with open(os.path.join(CFG.OUTPUT_DIR, "part1_manifest.json"), "w") as f:
    json.dump(final_manifest, f, indent=1)

print("\npart1_manifest.json saved. Part 1 complete.")


## Summary - Kaggle-limit safety checklist

- [x] No CNN forward/backward pass during training - only small cached feature vectors are read from
  `features.h5`.
- [x] Mixed precision (`torch.autocast` + `GradScaler`) reduces GPU memory usage and speeds up training.
- [x] Gradient clipping prevents exploding gradients from derailing training.
- [x] A checkpoint is saved every epoch and training automatically resumes from it, so a Kaggle session
  ending mid-training never loses progress - simply re-run the notebook.
- [x] Training stops gracefully with a safety margin before the configured wall-clock time budget is
  exceeded, rather than risking an abrupt kernel kill.
- [x] Early stopping avoids spending the time budget on epochs that no longer improve validation BLEU.
- [x] BLEU is computed on a subsample and only every few epochs, since greedy decoding is far more
  expensive than a forward pass.
- [x] Every long-running loop logs progress periodically to avoid a silent-cell kernel timeout.
- [x] All outputs are written to a small, fixed set of files, well within Kaggle's `/kaggle/working`
  quota.

**Next step:** Part 2 reuses the same cached spatial features (`features.h5`, already stored as
`num_tokens x feat_dim` rather than a single pooled vector) and replaces the LSTM decoder with a Transformer
decoder using self-attention, cross-attention to the image tokens, and positional encoding.
